# 🎹 Aria — Chopin LoRA (Merged) Generator

**Workflow:**
1. Run Step 1 (install)
2. Run Step 2 (upload LoRA zip)
3. Run Step 3 (merge at chosen strength)
4. Run Step 4 (load merged model)
5. Run Step 5 (generate) — rerun freely
6. Run Step 6 (listen)

**To try a different strength:** Run the Clear cell, then rerun Step 3 with new `LORA_STRENGTH`, then Step 4, then Step 5.

**Runtime:** T4 GPU

## Step 1 — Install
**Run once per session.**

In [1]:
#@title Install Dependencies
!pip install -q "torchao>=0.16.0"
!pip uninstall -q -y transformers
!pip install -q "git+https://github.com/EleutherAI/aria.git#egg=aria[all]"
!pip install -q huggingface_hub peft safetensors accelerate midi2audio
!apt-get install -qq fluidsynth
!wget -q -O /tmp/piano2.sf2 https://github.com/urish/cinto/raw/master/media/FluidR3_GM.sf2

import os, torch, gc, glob
from datetime import datetime
from aria.run import _get_prompt
from aria.inference.sample_cuda import sample_batch
from ariautils.tokenizer import AbsTokenizer

os.makedirs('aria_weights', exist_ok=True)
os.makedirs('generated_midi', exist_ok=True)

print(f'✅ Ready — GPU: {torch.cuda.get_device_name(0)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.1 MB/s eta 0:00:00
DEPRECATION: git+https://github.com/EleutherAI/aria.git#egg=aria[all] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

## Step 2 — Upload LoRA ZIP
**Upload your LoRA zip file once. No need to re-upload when changing strength.**

In [5]:
#@title Load LoRA from Google Drive
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

# List available checkpoints in your Drive folder
DRIVE_DIR = "/content/drive/MyDrive/chopin_lora_checkpoints_run2"
available = sorted([f for f in os.listdir(DRIVE_DIR) if f.endswith('.zip')])

print("Available checkpoints:")
for i, f in enumerate(available):
    print(f"  {i}: {f}")

# ← Change this to the number you want to load
CHOICE = 5

zip_path = os.path.join(DRIVE_DIR, available[CHOICE])
print(f"\nLoading: {available[CHOICE]}")

# Extract
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('lora_extracted')

# Find adapter_config.json
LORA_PATH = None
for root, dirs, filenames in os.walk('lora_extracted'):
    if 'adapter_config.json' in filenames:
        LORA_PATH = root
        break

if LORA_PATH is None:
    print('❌ adapter_config.json not found')
else:
    print(f'✅ LoRA ready at: {LORA_PATH}')
    print(f'Files: {os.listdir(LORA_PATH)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Available checkpoints:
  0: chopin_lora_step1200_val0.401_train0.418.zip
  1: chopin_lora_step1300_val0.476_train0.550.zip
  2: chopin_lora_step1500_val0.344_train0.351.zip
  3: chopin_lora_step1800_val0.300_train0.298.zip
  4: chopin_lora_step2450_val0.254_train0.253.zip
  5: chopin_lora_step2750_val0.246_train0.275.zip
  6: chopin_lora_step900_val0.376_train0.319.zip
  7: new_chopin_lora_step2750_val0.246_train0.275.zip

Loading: chopin_lora_step2750_val0.246_train0.275.zip
✅ LoRA ready at: lora_extracted
Files: ['tokenizer_config.json', 'adapter_model.safetensors', 'optimizer.pt', 'trainer_state.json', 'adapter_config.json', 'training_args.bin', 'README.md', 'tokenization_aria.py', 'scheduler.pt', 'rng_state.pth']


In [2]:
#@title Upload Local LoRA (Zip file)
from google.colab import files as colab_files
import zipfile

print('Upload your LoRA zip file...')
uploaded = colab_files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('lora_extracted')

LORA_PATH = None
for root, dirs, filenames in os.walk('lora_extracted'):
    if 'adapter_config.json' in filenames:
        LORA_PATH = root
        break

if LORA_PATH is None:
    print('❌ adapter_config.json not found — check your zip')
else:
    print(f'✅ LoRA ready at: {LORA_PATH}')
    print(f'Files: {os.listdir(LORA_PATH)}')

Upload your LoRA zip file...


KeyboardInterrupt: 

## Step 3 — Merge LoRA at Chosen Strength
**Change `LORA_STRENGTH` and rerun to try different values.**
Each strength is saved as a separate file.

In [6]:
from transformers import AutoModelForCausalLM, AutoConfig
from peft import PeftModel
import transformers.tokenization_utils as _tok_utils
from transformers.tokenization_utils_base import BatchEncoding
from safetensors.torch import load_file, save_file
from huggingface_hub import hf_hub_download

# ╔══════════════════════════════════════════╗
# ║              CONFIG                     ║
# ╚══════════════════════════════════════════╝

LORA_STRENGTH = 1   # @param {type:"number"} 0.5 = subtle | 1.0 = full | 1.5 = exaggerated

# ══════════════════════════════════════════

if not hasattr(_tok_utils, 'BatchEncoding'):
    _tok_utils.BatchEncoding = BatchEncoding

print('Loading base model + gen checkpoint...')
config = AutoConfig.from_pretrained('loubb/aria-medium-base', trust_remote_code=True)
if not hasattr(config, 'initializer_range'):
    config.initializer_range = 0.02

hf_base = AutoModelForCausalLM.from_pretrained(
    'loubb/aria-medium-base', config=config, trust_remote_code=True, dtype=torch.float16
)
gen_ckpt = hf_hub_download(repo_id='loubb/aria-medium-base', filename='model-gen.safetensors')
hf_base.load_state_dict(load_file(gen_ckpt), strict=False)

print(f'Applying LoRA at strength {LORA_STRENGTH}...')
lora_model = PeftModel.from_pretrained(hf_base, LORA_PATH)
for module in lora_model.modules():
    if hasattr(module, 'scaling') and isinstance(module.scaling, dict):
        for key in module.scaling:
            module.scaling[key] *= LORA_STRENGTH

print('Merging...')
merged = lora_model.merge_and_unload()

MERGED_PATH = f'aria_weights/chopin_s{LORA_STRENGTH}.safetensors'
save_file(merged.state_dict(), MERGED_PATH)

del merged, lora_model, hf_base
gc.collect()
torch.cuda.empty_cache()

print(f'✅ Saved: {MERGED_PATH}')
print(f'Now run Step 4 to load it.')

Loading base model + gen checkpoint...


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

configuration_aria.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/loubb/aria-medium-base:
- configuration_aria.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_aria.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/loubb/aria-medium-base:
- modeling_aria.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

model-gen.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

Applying LoRA at strength 1...
Merging...
✅ Saved: aria_weights/chopin_s1.safetensors
Now run Step 4 to load it.


In [ ]:
if 'LORA_PATH' not in dir() or LORA_PATH is None:
    print('⚠️  No LoRA loaded — run Step 2 first to upload your LoRA zip')
else:
    print(f'✅ LoRA found: {LORA_PATH}')
    print(f'   Files: {os.listdir(LORA_PATH)}')

## Step 4 — Load Merged Model
**Run after Step 3. Uses the merged file from the strength you just set.**

In [7]:
#@title Load LoRA
from aria.run import _load_inference_model_torch

model = _load_inference_model_torch(
    checkpoint_path=MERGED_PATH,
    config_name='medium',
    strict=False,
)
print(f'✅ Loaded: {MERGED_PATH}')
print(f'KV cache enabled — fast generation ready')

✅ Loaded: aria_weights/chopin_s1.safetensors
KV cache enabled — fast generation ready


## Step 5 — Upload Seed & Generate
**Rerun this cell freely with different settings.**

In [13]:
#@title Upload Seed
from google.colab import files as colab_files

print('Upload your seed MIDI file:')
uploaded = colab_files.upload()
SEED_MIDI_PATH = list(uploaded.keys())[0]
print(f'✅ Seed: {SEED_MIDI_PATH}')

Upload your seed MIDI file:


Saving scratch_s1.25_(t1.18_mp0.025)_2026-05-17_20-35-56_7 (orginal).mid to scratch_s1.25_(t1.18_mp0.025)_2026-05-17_20-35-56_7 (orginal).mid
✅ Seed: scratch_s1.25_(t1.18_mp0.025)_2026-05-17_20-35-56_7 (orginal).mid


**Generate from scratch Test1 - Methode forced Break**

In [ ]:
# ── Generate guaranteed new piece (seed sets key/style, then fresh start) ──
tokenizer = AbsTokenizer()
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

# ╔══════════════════════════════════════════╗
# ║              CONFIG                     ║
# ╚══════════════════════════════════════════╝

SEED_DURATION_SEC = 21      # @param {type:"number"}
MAX_NEW_TOKENS    = 2048    # @param {type:"number"} ~2-3 min of music
TEMPERATURE       = 0.98    # @param {type:"number"}
MIN_P             = 0.034   # @param {type:"number"}
NUM_VARIATIONS    = 6       # @param {type:"number"} generated in parallel — same time as 1!

seed_prompt = _get_prompt(SEED_MIDI_PATH, prompt_duration_s=SEED_DURATION_SEC)

# <E> = end current piece, <S> = begin new piece
new_piece_prompt = seed_prompt + [1, 0]  # <E> then <S>

print(f'Seed prompt: {len(seed_prompt)} tokens')
print(f'New piece prompt: {len(new_piece_prompt)} tokens (+<E><S>)')
print(f'Generating {NUM_VARIATIONS} fresh pieces in Chopin style...\n')

results = sample_batch(
    model=model,
    tokenizer=tokenizer,
    prompt=new_piece_prompt,
    num_variations=NUM_VARIATIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    temp=TEMPERATURE,
    min_p=MIN_P,
)

for idx, seq in enumerate(results):
    midi_dict = tokenizer.detokenize(seq)
    out_path = f'generated_midi2/newpiece_s{LORA_STRENGTH}_(t{TEMPERATURE}_mp{MIN_P})_{timestamp}_{idx+1}.mid'
    midi_dict.to_midi().save(out_path)
    print(f'✅ Saved: {out_path}')

**From Scratch - zero prompt**

In [17]:
# ── TRUE FROM-SCRATCH Generation ─────────────────────────────────────
# No seed MIDI — starts from a bare <S> token.
# All style comes purely from the merged Chopin LoRA weights.

tokenizer = AbsTokenizer()
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
os.makedirs('generated_midi_scratch', exist_ok=True)

# ╔══════════════════════════════════════════╗
# ║              CONFIG                     ║
# ╚══════════════════════════════════════════╝

MAX_NEW_TOKENS  = 512     # @param {type:"number"}
TEMPERATURE     = 1.1  # @param {type:"number"}
MIN_P           = 0.033  # @param {type:"number"}
NUM_VARIATIONS  = 10        # @param {type:"number"} all generated in parallel

# ══════════════════════════════════════════

# Token 0 = <S> (start-of-piece) — that's the entire prompt
scratch_prompt = [0]

print(f'🎹 Generating {NUM_VARIATIONS} pieces from SCRATCH (no seed MIDI)...')
print(f'   Strength={LORA_STRENGTH}  Temp={TEMPERATURE}  min_p={MIN_P}  max_tokens={MAX_NEW_TOKENS}\n')

results = sample_batch(
    model=model,
    tokenizer=tokenizer,
    prompt=scratch_prompt,
    num_variations=NUM_VARIATIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    temp=TEMPERATURE,
    min_p=MIN_P,
)

for idx, seq in enumerate(results):
    midi_dict = tokenizer.detokenize(seq)
    out_path = f'generated_midi_scratch/scratch_{timestamp}_s{LORA_STRENGTH}_(t{TEMPERATURE}_mp{MIN_P})_{idx+1}.mid'
    midi_dict.to_midi().save(out_path)
    print(f'  ✅ {out_path}')

print(f'\n🎶 Done! {NUM_VARIATIONS} original pieces — no seed used.')

🎹 Generating 10 pieces from SCRATCH (no seed MIDI)...
   Strength=1  Temp=1.1  min_p=0.033  max_tokens=512

Using hyperparams: temp=1.1, top_p=None, min_p=0.033, gen_len=512


  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_1.mid
  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_2.mid
  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_3.mid
  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_4.mid
  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_5.mid
  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_6.mid
  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_7.mid
  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_8.mid
  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_9.mid
  ✅ generated_midi_scratch/scratch_s1_(t1.1_mp0.033)_2026-05-25_00-08-55_10.mid

🎶 Done! 10 original pieces — no seed used.


**Contenuation Gen**

In [15]:
tokenizer = AbsTokenizer()

# ╔══════════════════════════════════════════╗
# ║              CONFIG                     ║
# ╚══════════════════════════════════════════╝

SEED_DURATION_SEC = 12  # @param {type:"number"}
MAX_NEW_TOKENS    = 1024   # @param {type:"number"}  ~2-3 min of music
TEMPERATURE       = 1.05  # @param {type:"number"}
MIN_P = 0.032  # @param {type:"number"}
NUM_VARIATIONS    = 10       # @param {type:"number"} generated in parallel — same time as 1!
DIR_PATH_MUSIC = 'generated_midi'  #@param {type:"string"}

# ══════════════════════════════════════════

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
prompt = _get_prompt(SEED_MIDI_PATH, prompt_duration_s=SEED_DURATION_SEC)
print(f'Generating {NUM_VARIATIONS} variations in parallel...\n')

results = sample_batch(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    num_variations=NUM_VARIATIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    temp=TEMPERATURE,
    min_p=MIN_P,
)

for idx, seq in enumerate(results):
    midi_dict = tokenizer.detokenize(seq)
    out_path = f'{DIR_PATH_MUSIC}/Aria_{timestamp}_(t{TEMPERATURE}_mp{MIN_P})_s{LORA_STRENGTH}_{idx+1}.mid'
    midi_dict.to_midi().save(out_path)
    print(f'✅ Saved: {out_path}')

Generating 10 variations in parallel...

Using hyperparams: temp=1.05, top_p=None, min_p=0.032, gen_len=1024


✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_1.mid
✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_2.mid
✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_3.mid
✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_4.mid
✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_5.mid
✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_6.mid
✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_7.mid
✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_8.mid
✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_9.mid
✅ Saved: generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_10.mid


## Step 6 — Listen

In [16]:
#@title  Midi Render

DIR_PATH_MUSIC = 'generated_midi'  #@param {type:"string"}

from IPython.display import Audio, display

for midi_path in sorted(glob.glob(f'{DIR_PATH_MUSIC}/*.mid')):
    wav_path = midi_path.replace('.mid', '.wav')
    mp3_path = midi_path.replace('.mid', '.mp3')
    os.system(f'fluidsynth -ni /tmp/piano2.sf2 "{midi_path}" -F "{wav_path}" -r 16000')
    os.system(f'ffmpeg -loglevel quiet -i "{wav_path}" "{mp3_path}"')
    print('🎹',midi_path)
    display(Audio(mp3_path))

🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_1.mid


🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_10.mid


🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_2.mid


🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_3.mid


🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_4.mid


🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_5.mid


🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_6.mid


🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_7.mid


🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_8.mid


🎹 generated_midi/Aria_(t1.05_mp0.032)_s1_2026-05-25_00-06-50_9.mid


---
## Utilities

**Delet Gen folder**

In [11]:
DIR_PATH_MUSIC = 'generated_midi_scratch'  #@param {type:"string"}
# ── Delete generated MIDIs ───────────────────────────────────────────
for f in glob.glob(f'{DIR_PATH_MUSIC}//*'):
    os.remove(f)
print('🗑️ Generated MIDIs cleared')

🗑️ Generated MIDIs cleared


In [4]:

# ── Delete Lora files ───────────────────────────────────────────
for f in glob.glob('/content/lora_extracted/*'):
    os.remove(f)
print('🗑️ Generated MIDIs cleared')

🗑️ Generated MIDIs cleared


**gpu clean model stay**

In [ ]:
import torch, gc

# Free all non-model tensors
torch.cuda.empty_cache()
gc.collect()

# Show what's left
print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

Allocated: 3.86 GB
Reserved:  6.01 GB


**Clear GPU Model to try new strenth lora**

In [ ]:
# ── Clear model from GPU — run before trying a new strength ──────────
# After running this: rerun Step 3 with new LORA_STRENGTH, then Step 4, then Step 5

try:
    del model
    print('Model cleared')
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print('✅ GPU cleared — rerun Step 3 with new LORA_STRENGTH')

Model cleared
✅ GPU cleared — rerun Step 3 with new LORA_STRENGTH


**Zip all gen foler to download**

In [ ]:
from google.colab import files as colab_files
import zipfile

zip_path = '/content/generated_midis.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in glob.glob('generated_midi/*.mid'):
        zipf.write(f)
print('✅ ZIP created')
colab_files.download(zip_path)

# Inpaint - Test

In [ ]:
!pip install -q pretty_midi
from ariautils.midi import MidiDict
import pretty_midi, torch, torch.nn.functional as F, gc
from datetime import datetime
